## Imports

In [64]:
from arcgis.gis import GIS
from arcgis.geocoding import geocode
from arcgis.geometry import Geometry, filters, Point, buffer, LengthUnits, AreaUnits, Polygon
from arcgis.features import FeatureLayer, FeatureSet
from arcgis.geometry.filters import intersects
import pandas as pd

## Functions

## Analysis

### Login

#### Local

In [ ]:
import yaml

In [2]:
with open("../../CityLogins.yaml", "r") as file:
    config = yaml.safe_load(file)

In [3]:
def get_gis(city_name):
    """
    Returns a connected GIS object.
    """
    city_config = config['cities'][city_name]
    url = city_config['url']
    username = city_config['username']
    password = city_config['password']
    return GIS(url, username, password)

In [6]:
source_city = "Abonmarche"
gis = get_gis(source_city)
print(f"Connected to the source GIS of {source_city}.")

Connected to the source GIS of Abonmarche.


#### Online

In [ ]:
gis = GIS("home")

### Make dataframe of BZA Form

In [7]:
BZA = gis.content.get("8fd6f6e7d1cc4c54afb5b1c4ece7eaf6")
BZA_table = BZA.tables[0]

In [11]:
BZA_df = pd.DataFrame.spatial.from_layer(BZA_table)

In [15]:
BZA_df

,OBJECTID,PetitionersName,Address,City,State,Zip,CaseNo,Request,Location,Zoning,...,SiteZip,Agent,AgentAddress,AgentCity,AgentState,AgentZip,MeetingDate,TimeOfMeeting,AppealDate,GlobalID
0,1,Garrick Garcia,95 W Main St,Benton Harbor,Michigan,49022,111-111,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,01/01/2025,12:00,<NA>,225bd803-f806-4b8b-8e3c-c648afe13b53
1,2,Jeff Weaver,250 W Main St,Benton Harbor,Michigan,49022,111-112,General,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1/2/25,2:00,<NA>,20ff9420-dfed-47b9-9fef-e8b35c5ed3fc


In [13]:
CaseNo = "111-111"

In [14]:
# filter the dataframe to get the row with the case number
BZA_df[BZA_df['CaseNo'] == CaseNo]

,OBJECTID,PetitionersName,Address,City,State,Zip,CaseNo,Request,Location,Zoning,...,SiteZip,Agent,AgentAddress,AgentCity,AgentState,AgentZip,MeetingDate,TimeOfMeeting,AppealDate,GlobalID
0,1,Garrick Garcia,95 W Main St,Benton Harbor,Michigan,49022,111-111,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,01/01/2025,12:00,<NA>,225bd803-f806-4b8b-8e3c-c648afe13b53


### Buffer the selected parcel

In [55]:
parcel_item = gis.content.get("ba795b77ce2249f8bf89c852bf5eb53a")
parcel_layer = parcel_item.layers[0]

In [56]:
parcelnumb = "11-54-0340-0147-01-0"
buffer_distance = 200

In [57]:
subject_parcel = parcel_layer.query(where=f"parcelnumb = '{parcelnumb}'").features[0].geometry

In [58]:
parcel_geom = Polygon(subject_parcel)

In [59]:
layer_extent = parcel_layer.properties.extent
layer_sr = layer_extent['spatialReference'] if layer_extent else None

In [60]:
layer_sr

{'wkid': 102100, 'latestWkid': 3857}

In [61]:
buffer_result = buffer(geometries =[parcel_geom], in_sr =layer_sr, distances=[buffer_distance], unit=LengthUnits.FOOT, geodesic=False)

In [ ]:
buffer_result

### Query neeighbor parcels

In [65]:
buffer_geometry = Geometry(buffer_result[0])

In [63]:
subject_parcel_geom = Geometry(subject_parcel)

In [67]:
neighbors_filter = intersects(geometry=buffer_geometry, sr=layer_sr) 

In [69]:
neighbors_df = parcel_layer.query(geometry_filter=neighbors_filter, as_df=True)

In [70]:
neighbors_df

,FID,geoid,sourceagen,parcelnumb,usecode,usedesc,zoning,zoning_des,struct,multistruc,...,taxable_22,homestead,vacant,cibecf,lat,lon,taxyear,Shape__Area,Shape__Length,SHAPE
0,1042,26021,,11-54-0340-0148-00-8,201,,,,0,0,...,26248.0,0,0,0,42.116116,-86.456398,,811.078125,128.009629,"{""rings"": [[[-9624283.2921116, 5178364.9925715..."
1,338,26021,,11-54-0340-0149-00-4,201,,,,0,0,...,128844.0,0,0,0,42.116057,-86.456647,,1776.425781,168.815635,"{""rings"": [[[-9624302.88434199, 5178358.914782..."
2,2325,26021,,11-54-0340-0052-01-9,202,,,,0,0,...,1539.0,0,0,0,42.116300,-86.455950,,635.949219,139.975066,"{""rings"": [[[-9624202.68566832, 5178422.514162..."
3,1500,26021,,11-51-0340-0208-07-4,201,,,,0,0,...,51396.0,0,0,0,42.115593,-86.455922,,566.273438,128.645758,"{""rings"": [[[-9624226.30766427, 5178283.175347..."
4,1014,26021,,11-54-0340-0049-01-8,201,,,,0,0,...,50825.0,0,0,0,42.116750,-86.455921,,1044.519531,146.616187,"{""rings"": [[[-9624254.11527307, 5178472.202413..."
5,1486,26021,,11-51-0340-0206-01-2,201,,,,0,0,...,44728.0,0,0,0,42.115665,-86.455613,,1097.332031,147.993581,"{""rings"": [[[-9624196.67441582, 5178292.644619..."
6,336,26021,,11-54-0340-0147-01-0,201,,,,0,0,...,35295.0,0,0,0,42.116144,-86.456252,,679.34375,120.12875,"{""rings"": [[[-9624263.68874928, 5178371.055357..."
7,1057,26021,,11-51-0340-0207-01-9,201,,,,0,0,...,278700.0,0,0,0,42.115614,-86.455833,,550.789062,127.912468,"{""rings"": [[[-9624216.2221184, 5178286.4018021..."
8,1501,26021,,11-51-0340-0208-08-2,201,,,,0,0,...,76747.0,0,0,0,42.115562,-86.456057,,1125.925781,149.632563,"{""rings"": [[[-9624235.76982099, 5178280.158988..."
9,2271,26021,,11-54-0013-0002-01-5,703,,,,0,0,...,0.0,0,0,0,42.116721,-86.456646,,11753.679688,429.482386,"{""rings"": [[[-9624337.33772439, 5178403.365272..."


### Format tables

In [71]:
# remove sublect parcel from neighbors_df by filtering out the parcelnumb
neighbors_df = neighbors_df[neighbors_df['parcelnumb'] != parcelnumb]

,FID,geoid,sourceagen,parcelnumb,usecode,usedesc,zoning,zoning_des,struct,multistruc,...,taxable_22,homestead,vacant,cibecf,lat,lon,taxyear,Shape__Area,Shape__Length,SHAPE
0,1042,26021,,11-54-0340-0148-00-8,201,,,,0,0,...,26248.0,0,0,0,42.116116,-86.456398,,811.078125,128.009629,"{""rings"": [[[-9624283.2921116, 5178364.9925715..."
1,338,26021,,11-54-0340-0149-00-4,201,,,,0,0,...,128844.0,0,0,0,42.116057,-86.456647,,1776.425781,168.815635,"{""rings"": [[[-9624302.88434199, 5178358.914782..."
2,2325,26021,,11-54-0340-0052-01-9,202,,,,0,0,...,1539.0,0,0,0,42.116300,-86.455950,,635.949219,139.975066,"{""rings"": [[[-9624202.68566832, 5178422.514162..."
3,1500,26021,,11-51-0340-0208-07-4,201,,,,0,0,...,51396.0,0,0,0,42.115593,-86.455922,,566.273438,128.645758,"{""rings"": [[[-9624226.30766427, 5178283.175347..."
4,1014,26021,,11-54-0340-0049-01-8,201,,,,0,0,...,50825.0,0,0,0,42.116750,-86.455921,,1044.519531,146.616187,"{""rings"": [[[-9624254.11527307, 5178472.202413..."
5,1486,26021,,11-51-0340-0206-01-2,201,,,,0,0,...,44728.0,0,0,0,42.115665,-86.455613,,1097.332031,147.993581,"{""rings"": [[[-9624196.67441582, 5178292.644619..."
7,1057,26021,,11-51-0340-0207-01-9,201,,,,0,0,...,278700.0,0,0,0,42.115614,-86.455833,,550.789062,127.912468,"{""rings"": [[[-9624216.2221184, 5178286.4018021..."
8,1501,26021,,11-51-0340-0208-08-2,201,,,,0,0,...,76747.0,0,0,0,42.115562,-86.456057,,1125.925781,149.632563,"{""rings"": [[[-9624235.76982099, 5178280.158988..."
9,2271,26021,,11-54-0013-0002-01-5,703,,,,0,0,...,0.0,0,0,0,42.116721,-86.456646,,11753.679688,429.482386,"{""rings"": [[[-9624337.33772439, 5178403.365272..."
10,2308,26021,,11-54-0340-0051-01-2,202,,,,0,0,...,1422.0,0,0,0,42.116431,-86.455915,,528.359375,123.174842,"{""rings"": [[[-9624202.71906417, 5178431.158189..."


In [89]:
# only keep columns parcelnumb, address, and owner
neighbors_df = neighbors_df[['parcelnumb', 'address', 'owner']]

In [96]:
filtered_BZA = BZA_df[BZA_df['CaseNo'] == CaseNo]

In [91]:
# remove the columns Address City State and Zip
filtered_BZA = filtered_BZA.drop(columns=['Address', 'City', 'State', 'Zip'])

In [92]:
# Ensure only one row is selected
if len(filtered_BZA) != 1:
    raise ValueError("The filtered BZA_df does not contain exactly one row. Check your CaseNo filter.")


In [93]:
# Get the row as a dictionary to add its columns and values
bza_row = filtered_BZA.iloc[0].to_dict()

In [94]:
neighbors_df = neighbors_df.copy()
# Broadcast the BZA values across all rows of neighbors_df
for col, value in bza_row.items():
    neighbors_df.loc[:, col] = value

In [95]:
neighbors_df

,parcelnumb,address,owner,OBJECTID,PetitionersName,CaseNo,Request,Location,Zoning,TaxID,...,SiteZip,Agent,AgentAddress,AgentCity,AgentState,AgentZip,MeetingDate,TimeOfMeeting,AppealDate,GlobalID
0,11-54-0340-0148-00-8,101 W MAIN ST,ARP GLOBAL HOLDINGS LLC,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53
1,11-54-0340-0149-00-4,115 W MAIN ST,LANDMARK PROPERTIES REDEVELOPMENT,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53
2,11-54-0340-0052-01-9,98 WATER ST,ARP GLOBAL HOLDINGS LLC,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53
3,11-51-0340-0208-07-4,90 W MAIN ST,CHANDLER GROUP LLC THE,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53
4,11-54-0340-0049-01-8,124 WATER ST,VICTORIA VENTURES LLC,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53
5,11-51-0340-0206-01-2,70 W MAIN ST,RENAISSANCE DEVELOPMENT FUND,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53
7,11-51-0340-0207-01-9,88 W MAIN ST,88 WEST MAIN STREET LLC,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53
8,11-51-0340-0208-08-2,92 W MAIN ST,RENAISSANCE DEVELOPMENT FUND,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53
9,11-54-0013-0002-01-5,0 COLFAX AVE,BENTON HARBOR CITY OF,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53
10,11-54-0340-0051-01-2,110 WATER ST,ARP GLOBAL HOLDINGS LLC,1,Garrick Garcia,111-111,None,None,None,None,...,None,None,None,None,None,None,01/01/2025,12:00,None,225bd803-f806-4b8b-8e3c-c648afe13b53


### Export CSV